# Backpropagation trong CNN thuần NumPy

> **Đồ án môn Toán cho Khoa học Máy tính** — áp dụng Đại số tuyến tính & Giải tích (Chain Rule) để tự viết Backpropagation bằng NumPy thuần.

| Phần | Nội dung |
|------|----------|
| Giới thiệu | Kiến trúc CNN & Forward Pass — dữ liệu chảy qua từng lớp |
| 1 | `Softmax.backward()` — điểm xuất phát của dòng gradient |
| 2 | `MaxPool2.backward()` — định tuyến gradient theo cơ chế winner-take-all |
| 3 | `Conv3x3.backward()` — thuật toán cốt lõi |
| 4 | Một vòng lặp huấn luyện hoàn chỉnh |
| 5 | Chứng minh tại sao $\partial L/\partial z_i = p_i - y_i$ |
| 6 | Gradient của Conv là một phép Convolution |
| 7 | Minh họa Loss giảm qua nhiều bước (Loss curve) |
| 8 | **Kết quả thực tế trên MNIST** — model đã train, accuracy thực |

**Dòng chảy gradient (Backward chain):**
```
Loss → Softmax.backward() → [d_pool]  shape (13,13,8)
                 ↓
            MaxPool2.backward() → [d_conv]  shape (26,26,8)
                         ↓
                    Conv3x3.backward() → None
```
Mỗi lớp nhận gradient từ lớp sau (`d_L_d_out`), cập nhật trọng số của mình, rồi trả về gradient cho lớp trước.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)

---
## Giới thiệu: Kiến trúc CNN & Forward Pass

Trước khi đi vào backward, hãy xem dữ liệu **chảy qua mạng như thế nào** ở forward pass — mỗi lớp biến đổi shape vì lý do toán học rõ ràng:

| Lớp | Input | Output | Toán học |
|-----|-------|--------|----------|
| **Conv3x3** (8 filters) | (28, 28) | (26, 26, 8) | $Y_{i,j,f} = \sum_{m,n} X_{i+m,j+n} \cdot W_{f,m,n}$ — tích chập |
| **MaxPool2** (2×2) | (26, 26, 8) | (13, 13, 8) | $Y_{i,j,f} = \max(\text{vùng } 2{\times}2)$ — gộp cực đại |
| **Flatten** | (13, 13, 8) | (1352,) | reshape — **Đại số tuyến tính**: vector hóa |
| **Softmax FC** | (1352,) | (10,) | $z = Wx + b,\; p_i = e^{z_i}/\sum_k e^{z_k}$ |

Conv mất 2 viền vì kernel 3×3 không padding → 28 - 2 = 26. MaxPool chia đôi cả 2 chiều → 26 // 2 = 13.

In [ ]:
import sys
sys.path.insert(0, '.')
from src.model import Conv3x3 as _Conv3x3, MaxPool2 as _MaxPool2, Softmax as _Softmax

np.random.seed(0)
_conv = _Conv3x3(num_filters=8)
_pool = _MaxPool2()
_sm   = _Softmax(input_len=13*13*8, nodes=10)

_image = np.random.rand(28, 28)   # ảnh placeholder — thay bằng MNIST thực ở Phần 8

x1 = _conv.forward(_image)        # (28,28) → (26,26,8)
x2 = _pool.forward(x1)            # (26,26,8) → (13,13,8)
x3 = _sm.forward(x2)              # flatten + FC + softmax → (10,)

print("Forward Pass — Shape transformation:")
print(f"  Input      : {_image.shape}     (ảnh grayscale, pixel ∈ [0,1])")
print(f"  → Conv3x3  : {x1.shape}  (26 = 28 - 2, không padding)")
print(f"  → MaxPool2 : {x2.shape}  (13 = 26 // 2)")
print(f"  → Flatten  : ({x2.flatten().shape[0]},)    (13 × 13 × 8 = 1352)")
print(f"  → Softmax  : {x3.shape}       (xác suất cho 10 chữ số 0–9)")
print(f"\nDự đoán (chưa train): chữ số {np.argmax(x3)}  — confidence {x3.max()*100:.1f}%")

# Hiển thị input và 8 feature maps sau Conv
fig, axes = plt.subplots(2, 5, figsize=(13, 5))
axes[0, 0].imshow(_image, cmap='gray')
axes[0, 0].set_title('Input (28×28)', fontsize=9); axes[0, 0].axis('off')
for f in range(8):
    r, c = divmod(f + 1, 5)
    axes[r, c].imshow(x1[:, :, f], cmap='RdBu_r')
    axes[r, c].set_title(f'Conv filter {f}', fontsize=9); axes[r, c].axis('off')
axes[1, 4].bar(range(10), x3, color='steelblue')
axes[1, 4].set_title('Softmax output (10,)', fontsize=9)
axes[1, 4].set_xticks(range(10))
plt.suptitle('Forward Pass — dữ liệu chảy qua từng lớp', fontsize=11)
plt.tight_layout(); plt.show()

---
## 1. Softmax + Cross-Entropy Backward

**Loss** (Cross-Entropy):
$$L = -\ln(p_{\text{đúng}})$$

Gradient khởi đầu (dL/dp):
$$\frac{\partial L}{\partial p_i} = \begin{cases} -1/p_i & i = \text{nhãn đúng} \\ 0 & \text{còn lại} \end{cases}$$

Kết hợp Jacobian của Softmax + Chain Rule, kết quả rút gọn cực đẹp:
$$\frac{\partial L}{\partial z_i} = p_i - y_i$$

Gradient theo FC weights, bias, và input:
$$\frac{\partial L}{\partial W} = x \otimes \frac{\partial L}{\partial z}, \quad
\frac{\partial L}{\partial b} = \frac{\partial L}{\partial z}, \quad
\frac{\partial L}{\partial x} = W \cdot \frac{\partial L}{\partial z}$$

In [ ]:
class Softmax:
    def __init__(self, input_len, nodes):
        self.weights = np.random.randn(input_len, nodes) / input_len
        self.biases  = np.zeros(nodes)

    def forward(self, input_volume):
        self.last_input_shape = input_volume.shape
        x = input_volume.flatten()
        self.last_input = x
        totals = np.dot(x, self.weights) + self.biases
        self.last_totals = totals
        exp = np.exp(totals - np.max(totals))   # numerical stability
        self.last_exp, self.last_sum = exp, exp.sum()
        return exp / self.last_sum

    def backward(self, d_L_d_out, lr):
        for i, grad in enumerate(d_L_d_out):
            if grad == 0:
                continue
            t_exp, S = self.last_exp, self.last_sum

            # Jacobian của Softmax tại hàng i
            d_out_d_t          = -t_exp[i] * t_exp / S**2
            d_out_d_t[i]       =  t_exp[i] * (S - t_exp[i]) / S**2

            d_L_d_t      = grad * d_out_d_t                         # dL/dz
            d_L_d_w      = self.last_input[np.newaxis].T @ d_L_d_t[np.newaxis]
            d_L_d_b      = d_L_d_t
            d_L_d_inputs = self.weights @ d_L_d_t

            self.weights -= lr * d_L_d_w
            self.biases  -= lr * d_L_d_b
            return d_L_d_inputs.reshape(self.last_input_shape)

In [ ]:
# Demo: 1 forward + 1 backward step
sm = Softmax(input_len=8, nodes=10)

x    = np.random.rand(2, 2, 2)   # giả lập output của MaxPool (shape nhỏ)
probs = sm.forward(x)

label = 3
loss  = -np.log(probs[label])

gradient        = np.zeros(10)
gradient[label] = -1 / probs[label]   # dL/dp (chỉ 1 phần tử khác 0)

d_back = sm.backward(gradient, lr=0.005)

print(f"Loss        : {loss:.4f}")
print(f"p (đúng)    : {probs[label]:.4f}")
print(f"dL/dx shape : {d_back.shape}")   # phải bằng shape của x

---
## 2. MaxPool2 Backward — Winner-take-all

MaxPool không có trọng số. Nhiệm vụ duy nhất: **định tuyến gradient** về đúng vị trí pixel đã "thắng" ở forward pass.

$$\frac{\partial L}{\partial X_{i,j,f}} = \begin{cases} \frac{\partial L}{\partial Y_{i',j',f}} & \text{nếu } X_{i,j,f} = \max(\text{vùng } 2{\times}2) \\ 0 & \text{còn lại} \end{cases}$$

Pixel không "thắng" → không ảnh hưởng output → gradient = 0.

In [ ]:
class MaxPool2:
    def iterate_regions(self, image):
        h, w, _ = image.shape
        for i in range(h // 2):
            for j in range(w // 2):
                yield image[i*2:i*2+2, j*2:j*2+2], i, j

    def forward(self, input_volume):
        self.last_input = input_volume
        h, w, f = input_volume.shape
        out = np.zeros((h // 2, w // 2, f))
        for region, i, j in self.iterate_regions(input_volume):
            out[i, j] = np.amax(region, axis=(0, 1))
        return out

    def backward(self, d_L_d_out, lr):
        d_L_d_input = np.zeros(self.last_input.shape)
        for region, i, j in self.iterate_regions(self.last_input):
            h2, w2, f = region.shape
            amax = np.amax(region, axis=(0, 1))
            for i2 in range(h2):
                for j2 in range(w2):
                    for f2 in range(f):
                        if region[i2, j2, f2] == amax[f2]:   # winner
                            d_L_d_input[i*2+i2, j*2+j2, f2] = d_L_d_out[i, j, f2]
        return d_L_d_input

In [ ]:
# Demo: xem gradient chảy về đúng vị trí max
pool = MaxPool2()

# shape (2, 2, 2) — ảnh 2×2, 2 filters
x4 = np.array([[[3., 1.],
                 [2., 4.]],
                [[5., 1.],
                 [0., 2.]]])

out  = pool.forward(x4)              # shape (1, 1, 2) — max của mỗi filter
print("Forward output (max):", out)

d_out = np.array([[[10., 20.]]])     # gradient đến từ phía sau
d_in  = pool.backward(d_out, lr=0)

print("\nGradient về input:")
print(d_in)
print("\n→ Filter 0: max là 5 (vị trí [1,0]) → gradient 10 chảy về đó")
print("→ Filter 1: max là 4 (vị trí [0,1]) → gradient 20 chảy về đó")

---
## 3. Conv3x3 Backward — Thuật toán cốt lõi

**Forward** — trượt filter $3{\times}3$ qua ảnh:
$$Y_{i,j,f} = \sum_{m=0}^{2}\sum_{n=0}^{2} X_{i+m,\,j+n} \cdot W_{f,m,n}$$

**Backward** — mỗi trọng số $W_{f,m,n}$ ảnh hưởng đến **tất cả** vị trí $(i,j)$ trên feature map:
$$\frac{\partial L}{\partial W_{f,m,n}} = \sum_{i}\sum_{j} X_{i+m,\,j+n} \cdot \frac{\partial L}{\partial Y_{i,j,f}}$$

> **Nhận xét:** Đạo hàm của phép chập cũng là một phép chập — giữa ảnh đầu vào $X$ và gradient $\partial L/\partial Y$.

Lớp Conv là lớp đầu tiên → không cần tính $\partial L / \partial X$, trả về `None`.

In [ ]:
class Conv3x3:
    def __init__(self, num_filters):
        self.num_filters = num_filters
        self.filters = np.random.randn(num_filters, 3, 3) / 9

    def iterate_regions(self, image):
        h, w = image.shape
        for i in range(h - 2):
            for j in range(w - 2):
                yield image[i:i+3, j:j+3], i, j

    def forward(self, image):
        self.last_input = image
        h, w = image.shape
        out = np.zeros((h - 2, w - 2, self.num_filters))
        for region, i, j in self.iterate_regions(image):
            out[i, j] = np.sum(region * self.filters, axis=(1, 2))
        return out

    def backward(self, d_L_d_out, lr):
        d_L_d_filters = np.zeros(self.filters.shape)   # (num_filters, 3, 3)

        for region, i, j in self.iterate_regions(self.last_input):
            for f in range(self.num_filters):
                # region: (3,3)  d_L_d_out[i,j,f]: scalar
                # → cộng dồn gradient (3,3) cho filter f tại vị trí (i,j)
                d_L_d_filters[f] += d_L_d_out[i, j, f] * region

        self.filters -= lr * d_L_d_filters
        return None   # Conv là lớp đầu tiên, không cần truyền gradient về phía trước

In [ ]:
# Demo: kiểm tra gradient numerically (finite difference)
conv = Conv3x3(num_filters=2)
image = np.random.rand(6, 6)         # ảnh 6×6 giả lập

out = conv.forward(image)            # (4, 4, 2)
d_L_d_out = np.random.rand(*out.shape)  # gradient giả từ lớp sau

# Lưu filter gốc trước khi backward
filters_before = conv.filters.copy()

conv.backward(d_L_d_out, lr=0.01)

print("Filter 0 — trước backward:\n", filters_before[0].round(4))
print("\nFilter 0 — sau backward:\n",  conv.filters[0].round(4))
print("\nDelta (= -lr * gradient):\n", (conv.filters[0] - filters_before[0]).round(6))

---
## 4. Một vòng lặp huấn luyện hoàn chỉnh

```
Forward:  image → Conv3x3 → MaxPool2 → Softmax → probs
Loss:     L = -log(probs[label])
Backward: Softmax → MaxPool → Conv  (gradient chảy ngược)
```

In [ ]:
# Pipeline đầy đủ trên 1 ảnh giả lập 28×28
conv    = Conv3x3(num_filters=8)
pool    = MaxPool2()
softmax = Softmax(input_len=13*13*8, nodes=10)

image = np.random.rand(28, 28)
label = 7
LR    = 0.005

# --- Forward ---
out   = conv.forward(image)      # (26, 26, 8)
out   = pool.forward(out)        # (13, 13, 8)
probs = softmax.forward(out)     # (10,)

loss  = -np.log(probs[label])
print(f"Trước backward — Loss: {loss:.4f}, dự đoán: {np.argmax(probs)}")

# --- Backward ---
gradient        = np.zeros(10)
gradient[label] = -1 / probs[label]

grad = softmax.backward(gradient, LR)   # cập nhật weights, biases
grad = pool.backward(grad, LR)          # định tuyến gradient
conv.backward(grad, LR)                 # cập nhật filters

# --- Forward lần 2 để xem loss thay đổi ---
out2   = conv.forward(image)
out2   = pool.forward(out2)
probs2 = softmax.forward(out2)
loss2  = -np.log(probs2[label])
print(f"Sau  backward — Loss: {loss2:.4f}, dự đoán: {np.argmax(probs2)}")
print(f"\n→ Loss giảm {loss - loss2:.4f} sau 1 bước Gradient Descent")

---
## 5. Chứng minh: Tại sao $\partial L / \partial z_i = p_i - y_i$?

**Bước 1** — Đạo hàm Softmax (Jacobian):
$$\frac{\partial p_i}{\partial z_j} = \begin{cases} p_i(1 - p_i) & i = j \\ -p_i p_j & i \neq j \end{cases}$$

**Bước 2** — Chain Rule với Cross-Entropy (chỉ nhãn đúng $c$ khác 0):
$$\frac{\partial L}{\partial z_j} = \sum_i \frac{\partial L}{\partial p_i} \cdot \frac{\partial p_i}{\partial z_j} = p_j - y_j$$

Kết quả rút gọn cực đẹp: gradient chỉ là **xác suất dự đoán trừ nhãn thật**.

In [ ]:
z = np.array([1.2, 0.5, -0.3, 2.1, 0.8])
label_demo = 3

exp_z = np.exp(z - np.max(z))
p     = exp_z / exp_z.sum()

# Jacobian đầy đủ (5×5)
J = np.zeros((len(p), len(p)))
for i in range(len(p)):
    for j in range(len(p)):
        J[i, j] = p[i] * ((i == j) - p[j])

dL_dp = np.zeros(len(p))
dL_dp[label_demo] = -1 / p[label_demo]

dL_dz_full   = J.T @ dL_dp
dL_dz_simple = p.copy()
dL_dz_simple[label_demo] -= 1

print('dL/dz (Jacobian đầy đủ):', dL_dz_full.round(6))
print('dL/dz (p_i - y_i):      ', dL_dz_simple.round(6))
print(f'Max diff: {np.abs(dL_dz_full - dL_dz_simple).max():.2e}  → Hai cách tính BẰNG NHAU')

---
## 6. Gradient của Conv là một phép Convolution

$$\frac{\partial L}{\partial W_{m,n}} = \sum_{i,j} X_{i+m,j+n} \cdot \frac{\partial L}{\partial Y_{i,j}}$$

Đây **chính xác** là phép chập giữa $X$ và $\partial L/\partial Y$ — đạo hàm của convolution cũng là một convolution.

In [ ]:
X = np.array([[1., 2., 3.],
              [4., 5., 6.],
              [7., 8., 9.]])
W = np.array([[1., 0.],
              [0., 1.]])

Y = np.array([[np.sum(X[i:i+2, j:j+2] * W) for j in range(2)] for i in range(2)])
print('Forward output Y:')
print(Y)

dL_dY = np.array([[1., 2.],
                  [3., 4.]])

# Chain Rule
dL_dW_chain = np.zeros((2, 2))
for i in range(2):
    for j in range(2):
        dL_dW_chain += dL_dY[i, j] * X[i:i+2, j:j+2]

# Convolution(X, dL_dY)
dL_dW_conv = np.array([[np.sum(X[m:m+2, n:n+2] * dL_dY) for n in range(2)] for m in range(2)])

print('dL/dW qua Chain Rule:')
print(dL_dW_chain)
print('dL/dW như phép Convolution(X, dL/dY):')
print(dL_dW_conv)
print(f'→ Hai cách tính BẰNG NHAU (diff = {np.abs(dL_dW_chain - dL_dW_conv).max():.0e})')

---
## 7. Minh họa: Loss giảm đơn điệu qua nhiều bước

Chứng minh Loss giảm liên tục khi Gradient Descent lặp đủ nhiều bước trên cùng một ảnh.

In [ ]:
# === THAY ĐỔI Ở ĐÂY ===
NUM_STEPS = 150
LR        = 0.005
LABEL     = 3
# ======================

np.random.seed(42)
conv_t  = Conv3x3(num_filters=8)
pool_t  = MaxPool2()
sm_t    = Softmax(input_len=13*13*8, nodes=10)
image_t = np.random.rand(28, 28)

losses, confidences = [], []
for step in range(NUM_STEPS):
    out   = conv_t.forward(image_t)
    out   = pool_t.forward(out)
    probs = sm_t.forward(out)
    losses.append(-np.log(probs[LABEL]))
    confidences.append(probs[LABEL] * 100)
    g = np.zeros(10); g[LABEL] = -1 / probs[LABEL]
    g = sm_t.backward(g, LR)
    g = pool_t.backward(g, LR)
    conv_t.backward(g, LR)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(losses, color='tomato')
ax1.set_title('Cross-Entropy Loss'); ax1.set_xlabel('Bước'); ax1.grid(alpha=0.3)
ax2.plot(confidences, color='steelblue')
ax2.set_title(f'Confidence cho chữ số {LABEL} (%)')
ax2.set_xlabel('Bước'); ax2.set_ylim(0, 100); ax2.grid(alpha=0.3)
plt.suptitle(f'Mini Training — LR={LR}, Label={LABEL}')
plt.tight_layout(); plt.show()

print(f'Loss:       {losses[0]:.4f} -> {losses[-1]:.4f}')
print(f'Confidence: {confidences[0]:.1f}% -> {confidences[-1]:.1f}%')

---
## 8. Kết quả thực tế trên MNIST

Backpropagation không chỉ đúng về lý thuyết — đây là bằng chứng thực nghiệm: model đã được train trên 1000 ảnh MNIST thực và lưu lại trọng số.

- **Xanh lá** = dự đoán đúng
- **Đỏ** = dự đoán sai
- Số trong ngoặc = confidence của model

In [ ]:
import sys
sys.path.insert(0, '.')
from src.model import load_model, predict as cnn_predict
from src.data import load_mnist_data

# Load model đã train và 200 ảnh test
conv_r, pool_r, sm_r = load_model('output/model/cnn_weights.npz')
_, _, test_images, test_labels = load_mnist_data(num_train=1, num_test=200)

# Đánh giá accuracy
preds = []
for img, lbl in zip(test_images, test_labels):
    pred, conf, _ = cnn_predict(conv_r, pool_r, sm_r, img)
    preds.append((pred, conf, lbl))

accuracy = sum(p == l for p, _, l in preds) / len(preds) * 100
print(f'\nAccuracy trên {len(preds)} ảnh test thực tế: {accuracy:.1f}%')
print(f'(README kỳ vọng ~81% sau 3 epoch trên 1000 ảnh train)')

# Hiển thị 10 mẫu — xanh = đúng, đỏ = sai
fig, axes = plt.subplots(2, 5, figsize=(13, 5))
for idx, ax in enumerate(axes.flat):
    pred, conf, true = preds[idx]
    ax.imshow(test_images[idx], cmap='gray')
    color = 'green' if pred == true else 'red'
    ax.set_title(f'Dự đoán: {pred} ({conf*100:.0f}%)\nThực tế : {true}',
                 color=color, fontsize=8)
    ax.axis('off')
plt.suptitle(f'10 mẫu dự đoán trên MNIST thực tế — Accuracy: {accuracy:.1f}%', fontsize=11)
plt.tight_layout(); plt.show()